# 07 · Mesh textures

Textures for the pure mesh avatar, baked with and without lining the frames up by optical flow (`training/flow.py`). For comparison, a plain bake with the old split between broad colour and detail (`detail_scale` 2 px instead of the config's), which averaged creases, folds and the ears' shape over all frames.

The frames disagree by about a centimetre about where each mole or crease is: the fits are that close, and the generated videos are not quite rigid. Baked from many frames, the detail averages away. Flow finds, for every pixel, where the frame shows what the textured mesh draws there; the textures are baked again reading the frames there, and that repeats for a few rounds.

Needs `go-task fit`. If `go-task train` has run, its colour correction between the videos is used too. Takes a few minutes on the GPU; nothing is written to `outputs/`.

In [ ]:
import dataclasses
import time

import matplotlib.pyplot as plt
import numpy as np
import torch

from avatar_splats.body_models.hybrid import Avatar
from avatar_splats.config import load_config
from avatar_splats.fitting import fitted_path
from avatar_splats.training import (
    ColourCorrection,
    ColourRecord,
    Textures,
    bake_textures,
    load_posing,
    load_views,
    scored_pixels,
    target,
    trained_path,
)
from avatar_splats.training.flow import draw, flow_bake
from avatar_splats.training.losses import psnr
from avatar_splats.training.train import training_views

ROUNDS = 2  # rounds of measuring the flow and baking again

config = load_config("../configs/default.yaml")
settings = config.training
device = settings.device
avatar = Avatar.load(fitted_path(config, "avatar"))
views = load_views(config)
posing = load_posing(config, avatar)
training = training_views(views, settings)
held_out = [view for view in views if view not in training]

colours_path = trained_path(config, "colours")
correction = (
    ColourCorrection.from_record(ColourRecord.load(colours_path)).to(device)
    if colours_path.exists()
    else None
)
print(f"{len(training)} training and {len(held_out)} held-out frames")

## Baking

In [ ]:
start = time.time()
narrow = Textures(
    bake_textures(
        avatar, training, posing, dataclasses.replace(settings, detail_scale=2.0), correction
    )
)
plain = Textures(bake_textures(avatar, training, posing, settings, correction))
print(f"plain bake: {time.time() - start:.0f} s")

start = time.time()
record, flows = flow_bake(avatar, training, posing, settings, ROUNDS, correction)
flowed = Textures(record)
print(f"flow bake, {ROUNDS} rounds: {time.time() - start:.0f} s")

## How far the frames were moved

The length of the last round's flow, in pixels of the training frames, over the pixels it was measured on. Most should be small; a long tail means the fits are far off somewhere.

In [ ]:
figure, axes = plt.subplots(1, 3, figsize=(14, 3))
for ax, video in zip(axes, sorted({view.video for view in training}), strict=True):
    lengths = (
        torch.cat(
            [
                flow.norm(dim=-1)[flow.norm(dim=-1) > 0]
                for (v, _), flow in flows.items()
                if v == video
            ]
        )
        .cpu()
        .numpy()
    )
    ax.hist(lengths, bins=60, range=(0, 20))
    ax.set_title(
        f"{video}: median {np.median(lengths):.1f} px, p90 {np.quantile(lengths, 0.9):.1f} px"
    )
    ax.set_xlabel("flow length (px)")

## Detail

Each row: the frame, the mesh baked with the 2 px split, the mesh with the plain textures, the mesh with the flow-baked ones, all drawn in the frame's fitted pose. Training frames first, then held-out ones the textures never saw. Look for moles, creases, eyebrows and lips.

In [ ]:
def crop(image, view, height, half=128):
    """A square around the performer's middle at ``height`` (0 top, 1 bottom) of her mask."""
    rows, columns = torch.nonzero(view.mask > 0.5, as_tuple=True)
    y = int(rows.min() + height * (rows.max() - rows.min()))
    x = int(columns.float().mean())
    return image[max(y - half, 0) : y + half, max(x - half, 0) : x + half]


def show(chosen):
    figure, axes = plt.subplots(len(chosen), 4, figsize=(16, 4 * len(chosen)), squeeze=False)
    for row, (view, height) in zip(axes, chosen, strict=True):
        frame = view.image.float().cpu().numpy() / 255
        images = [frame] + [
            draw(avatar, posing, textures, view).image.clamp(0, 1).cpu().numpy()
            for textures in (narrow, plain, flowed)
        ]
        for ax, image, title in zip(
            row, images, ("frame", "2 px split", "plain bake", "flow bake"), strict=True
        ):
            ax.imshow(crop(image, view, height))
            ax.set_title(f"{view.video} {view.frame}: {title}")
            ax.axis("off")


def pick(group, video, index):
    return next(view for view in group if view.video == video and view.index == index)


show([
    (pick(training, "body", 21), 0.55),
    (pick(training, "body", 41), 0.35),
    (pick(training, "head_static", 1), 0.45),
    (pick(training, "head_expressions", 25), 0.5),
])  # fmt: skip

In [ ]:
show([(view, 0.55 if view.video == "body" else 0.45) for view in held_out[::6]])

## Held-out frames

PSNR in dB of the textured mesh alone, in each held-out frame's fitted pose, where the performer is. The held-out frames are not lined up by flow, so this measures the textures as an avatar posed by the fits would show them: sharper detail in slightly the wrong place can even score a little lower.

In [ ]:
black = torch.zeros(3, device=device)
with torch.no_grad():
    for name, textures in (
        ("2 px split", narrow),
        ("plain bake", plain),
        ("flow bake", flowed),
    ):
        scores = {}
        for view in held_out:
            mesh = draw(avatar, posing, textures, view)
            image = (
                mesh.image if correction is None else correction(view.video, mesh.image, mesh.mask)
            )
            score = psnr(image.clamp(0, 1), target(view, black), scored_pixels(view, mesh.mask))
            scores.setdefault(view.video, []).append(score)
        print(name, {video: round(float(np.mean(s)), 2) for video, s in scores.items()})

## The textures

In [ ]:
figure, axes = plt.subplots(2, 2, figsize=(14, 14))
for column, (name, textures) in enumerate((("plain", plain), ("flow", flowed))):
    body = textures.body.cpu().numpy()
    face = textures.head.cpu().numpy()[:, : textures.head.shape[0]]
    axes[0, column].imshow(body[768:1280, 768:1280])
    axes[0, column].set_title(f"{name}: body texture, middle")
    axes[1, column].imshow(face)
    axes[1, column].set_title(f"{name}: face")
for ax in axes.flat:
    ax.axis("off")